# 07 · Baselines - why "tomorrow equals today" is so hard to beat

**Goal:** implement the simple forecasts ourselves, evaluate them on the validation and test windows, and understand *when* each works.

A baseline is not a throwaway. It is the answer to "what would we have done without machine learning?", and a complex model is only worth its complexity if it beats the **strongest** baseline by a meaningful, repeatable margin.

In [1]:
from tutorial_utils import *
style()
rl = load_panel("realistic")
feats = {h: build_features(rl, h) for h in HORIZONS}
f = feats[2]
print("columns available for baselines:", [c for c in f.columns if c in ("ref_price", "ma4", "ma8", "price_l52h")])

columns available for baselines: ['ref_price', 'ma4', 'ma8', 'price_l52h']


## The four baselines
| Baseline | Forecast for week *t + h* |
|---|---|
| **Naive** | the latest available (cleaned) price at the origin |
| **Seasonal naive** | the price one year before the *target* week (`t + h − 52`); falls back to naive if unavailable |
| **Moving average (k)** | the average of the last *k* prices (k = 4 or 8) |

All four use the *same* cleaned, as-known price series (notebook 03) and are scored on the *same* rows, so the comparison is fair.

In [2]:
def baseline_preds(ev):
    return {"Naive": ev.ref_price, "Seasonal naive": ev.price_l52h.fillna(ev.ref_price),
            "MA(4)": ev.ma4.fillna(ev.ref_price), "MA(8)": ev.ma8.fillna(ev.ref_price)}

def baseline_table(lo, hi):
    out = {}
    for h in HORIZONS:
        ev = feats[h][(feats[h].origin_week.between(lo, eval_hi(h, hi))) & feats[h].ref_price.notna() & feats[h].y_obs.notna()]
        ev = ev[~ev.y_suspect]
        out[h] = {k: mae(ev.y_obs, v.loc[ev.index]) for k, v in baseline_preds(ev).items()}
    t = pd.DataFrame(out); t["mean"] = t.mean(axis=1); return t

val_t, test_t = baseline_table(TRAIN_END + 1, VAL_END), baseline_table(VAL_END + 1, TEST_END)
print("MAE (KES/kg) - VALIDATION window"); print(val_t.round(2).to_string())
print("\nMAE (KES/kg) - TEST window"); print(test_t.round(2).to_string())
print("\nskill vs naive (%) - TEST window"); print(((1 - test_t / test_t.loc["Naive"]) * 100).round(1).to_string())

MAE (KES/kg) - VALIDATION window
                   1     2     4  mean
Naive           3.64  4.86  6.46  4.99
Seasonal naive  8.45  8.41  8.29  8.38
MA(4)           4.53  5.42  6.76  5.57
MA(8)           5.43  6.19  7.51  6.38

MAE (KES/kg) - TEST window
                    1      2      4   mean
Naive            4.14   5.88   8.47   6.16
Seasonal naive  10.89  11.14  11.74  11.25
MA(4)            5.61   7.02   9.54   7.39
MA(8)            7.51   8.94  11.57   9.34

skill vs naive (%) - TEST window
                    1     2     4  mean
Naive             0.0   0.0   0.0   0.0
Seasonal naive -163.1 -89.2 -38.5 -82.5
MA(4)           -35.6 -19.2 -12.6 -19.8
MA(8)           -81.5 -51.9 -36.5 -51.5


**What the table says**
* **The moving average is worse than naive at every horizon.** Averaging past prices adds *lag*; prices here are persistent, so the newest value is the best simple summary (notebook 01).
* **The seasonal naive forecast is far worse** - about 1.7-1.8 times naive's error. A price a year ago carries the seasonal pattern, but also last year's *random shocks*, and shocks are the main source of movement.
  Seasonal naive makes sense only when the seasonal signal is strong relative to everything else. Is it, for any crop?

## Seasonal naive by crop
We measure how much of each crop's price variation is a repeating annual pattern: regress the (de-meaned) log price on an annual and a semi-annual sine/cosine pair and look at R². We use the *clean* scenario so noise doesn't blur the answer.

In [3]:
clean = load_panel("clean"); lpc = np.log(wide(clean, "wholesale_price_kes_kg")); lpc = lpc - lpc.mean()
doy = (START_DATE + pd.to_timedelta((lpc.index.values - 1) * 7 + 3, unit="D")).dayofyear.values.astype(float)
X = np.column_stack([np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25), np.sin(4 * np.pi * doy / 365.25), np.cos(4 * np.pi * doy / 365.25)])
r2 = {}
for c in lpc.columns:
    yv = lpc[c].values; beta, *_ = np.linalg.lstsq(X, yv, rcond=None)
    r2[c] = 1 - ((yv - X @ beta) ** 2).sum() / (yv ** 2).sum()
crop_r2 = pd.Series(r2).groupby(lambda s: CROPS[s.split("|")[1]]).mean().rename("seasonal R²")

def crop_table(lo, hi):
    rows = []
    for h in HORIZONS:
        ev = feats[h][(feats[h].origin_week.between(lo, eval_hi(h, hi))) & feats[h].ref_price.notna() & feats[h].y_obs.notna()]
        ev = ev[~ev.y_suspect].copy()
        for k, v in baseline_preds(ev).items():
            ev[k] = (ev.y_obs - v).abs()
        rows.append(ev.groupby("crop_id")[["Naive", "Seasonal naive", "MA(4)", "MA(8)"]].mean().assign(h=h))
    return pd.concat(rows).groupby("crop_id").mean().drop(columns="h").rename(index=CROPS)
ct = crop_table(VAL_END + 1, TEST_END)
ct["seasonal naive skill %"] = (1 - ct["Seasonal naive"] / ct["Naive"]) * 100
ct.join(crop_r2).round(2)

,Naive,Seasonal naive,MA(4),MA(8),seasonal naive skill %,seasonal R²
crop_id,,,,,,
Maize,2.24,1.61,2.99,4.15,28.23,0.85
Tomatoes,14.53,27.47,16.11,18.64,-89.07,0.37
Irish potatoes,4.30,9.07,6.08,8.49,-111.14,0.64
Onions,3.74,7.28,4.57,6.34,-94.77,0.69


**Seasonal naive wins for exactly one crop - maize - which is also the crop with by far the strongest seasonal pattern** (seasonal R² ≈ 0.85 versus 0.37-0.69 for the others), the lowest volatility and a very slow-moving price.
This is a perfectly valid scientific result, and it has a practical lesson: *the right baseline depends on the crop*. A forecasting service that compared everything to naive only would have missed that, for maize, the "seasonal naive" is the
strongest baseline a model must beat.

## The pipeline's numbers
Compare our hand-built baselines with the pipeline's table (`results/core_eight_MAE_test.csv`):

In [4]:
ref = pd.read_csv(RESULTS / "core_eight_MAE_test.csv", index_col=0).loc[["Naive", "Seasonal naive", "Moving average (4w)"]]
ours = test_t.loc[["Naive", "Seasonal naive", "MA(4)"]].copy(); ours.index = ref.index
cmp = pd.concat({"tutorial": ours.rename(columns=str), "pipeline": ref.rename(columns={"mean_over_horizons": "mean"})}, axis=1)
cmp.round(2)

tutorial                      pipeline                \
                           1      2      4   mean        1      2      4   
variant                                                                    
Naive                   4.14   5.88   8.47   6.16     4.14   5.88   8.47   
Seasonal naive         10.89  11.14  11.74  11.25    10.89  11.14  11.74   
Moving average (4w)     5.61   7.02   9.54   7.39     5.61   7.02   9.54   

                            
                      mean  
variant                     
Naive                 6.16  
Seasonal naive       11.25  
Moving average (4w)   7.39

The numbers agree to the second decimal - a good sign that the tutorial's simplified logic is faithful for these baselines. If they ever disagree materially, the *pipeline* is the reference and this tutorial needs fixing.

## Skill versus naive: a worked example
`skill = 1 − MAE_model / MAE_naive`. If naive has MAE 5.88 and a model has 5.25, skill is 1 − 5.25/5.88 = **10.7%**. Skill is the number we will track from here on. Three reasons:
1. it is scale-free, so crops and horizons can be compared;
2. it directly answers "is the model better than doing nothing?";
3. a skill of 5% is small *in absolute terms* but, as notebook 09 shows, can be statistically indistinguishable from zero with a short test window.

## Exercises
1. Add a baseline: the **median** of the last 4 prices. Is it better than the mean? Why might it be more robust to a reporting error that cleaning missed?
2. Add a **drift** baseline: `latest + h × (average weekly change over the last 13 weeks)`. Does it beat naive for any crop at h = 4?
3. Use the *raw* (uncleaned) prices as the baseline reference (`build_features(rl, h, handling=False)`). How much worse do all baselines become? Which suffers most?
4. For which crop-market pairs does the seasonal naive beat the naive on the **validation** window? Does the pattern hold in the test window?

**Where this lives in the pipeline:** `baseline_predictions` and the SARIMA/ARIMA functions in `src/agriprice/models.py`; the baseline columns are built in `src/agriprice/features.py` (`FeatureStore.frame`).